In [ ]:
!pip install -q numpy==2.4.4 2>/dev/null
!pip install -q onnx==1.21.0 2>/dev/null

<p style="background-color:#e6f7ff; 
          padding:15px; 
          color:#111;
          font-size:16px;
          border-width:3px; 
          border-color:#d0eefc; 
          border-style:solid;
          border-radius:6px">
✂️ <strong>Prune Unused Initializers</strong><br><br>
ONNX networks sometimes contain <code>initializers</code> (constant tensors) that no node actually reads. The NeuroGolf scorer counts <strong>every</strong> initializer element as a parameter — whether used or not:<br><br>
<code>cost = total_parameters + total_memory_in_bytes</code><br>
<code>score = max(1, 25 - ln(cost))</code><br><br>
An unused <code>[30]</code> tensor silently adds <strong>30</strong> to the cost and lowers the score for <em>zero benefit</em>.<br><br>
<strong>What this notebook does:</strong><br>
1️⃣ Loads each <code>.onnx</code> file from the submission<br>
2️⃣ Finds initializers <em>not referenced</em> by any node<br>
3️⃣ Removes them and re-saves<br>
4️⃣ Re-zips for submission<br><br>
No network behavior changes. Same outputs, fewer params, <strong>higher score</strong>. Free points. 🎯
</p>


$$ $$

## 🧬 Load Data

In [ ]:
!rm -r submission
!cp /kaggle/input/notebooks/anazemcev/neurogolf-2026-cost-optimal-blend/submission.zip /kaggle/working/submission.zip
!unzip -q /kaggle/working/submission.zip -d /kaggle/working/submission
!rm /kaggle/working/submission.zip 

In [ ]:
ls -la

$$ $$

## 📦 Imports

In [ ]:
import glob, os
from tqdm.auto import tqdm

import numpy as np
import onnx
from onnx import numpy_helper

$$ $$

## 🧹 Prune

In [ ]:
def cleanup_unused_initializers(task_num, model):
    """Remove initializers not referenced by any node. Returns cleaned model."""
    graph = model.graph
    
    used = set()
    for node in graph.node:
        used.update(node.input)
    
    to_remove = []
    for init in graph.initializer:
        if init.name not in used:
            to_remove.append(init)
            
    if not to_remove:
        return None
        
    print(f"[Task {task_num:03d}] Removing unused:")
    for init in to_remove:
        graph.initializer.remove(init)
        dims = list(init.dims) if init.dims else [1]  # Handle scalar
        print(f"  '{init.name}' {dims} = {int(np.prod(dims))} params saved")
    print()

    return model



def cleanup_all_tasks(task_dir="/kaggle/working/submission"):
    """Clean all .onnx files in a directory."""
    

    task_paths = sorted(glob.glob(f"{task_dir}/task*.onnx"))
    for path in tqdm(task_paths):
        name = os.path.basename(path)
        task_num = int(name.split("/")[-1].split(".")[0].replace("task",""))
        model = onnx.load(path)
        model = cleanup_unused_initializers(task_num, model)
        if model is not None:
            onnx.checker.check_model(model)
            onnx.save(model, path)

In [ ]:
cleanup_all_tasks(task_dir="/kaggle/working/submission")

$$ $$

## 📤 Submit

In [ ]:
import zipfile
import os

source_folder = '/kaggle/working/submission' 
output_filename = '/kaggle/working/submission.zip'

with zipfile.ZipFile(output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(source_folder):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, os.path.relpath(file_path, os.path.join(source_folder, '..')))

In [ ]:
ls